# 05 · Final model, external evaluation and 3D output for ArcGIS Pro

**Brief sections 10, 12 and 14.** The steps run in this order, and the order matters:

1. **Final model.** Take the model and training extent selected in notebook 04 by the pre-declared rule. Tune it once more on *all* development data with the same inner procedure: groups are the five outer folds, and 5 km ring boreholes are always in training. Refit on all development data.
2. **Freeze.** Save the model, its parameters, threshold and settings, and record the SHA-256 of the saved model file.
3. **External evaluation, once.** Only now are the labels of the external boreholes read. The result is stored with the model hash; re-running with a different model is refused unless explicitly allowed, and any such re-evaluation must be disclosed. In **smoke mode** the external boreholes are never touched: the same code runs on a stand-in set of development boreholes.
4. **3D prediction** on a regular grid inside the Downtown polygon, with flags for horizontal extrapolation (beyond the distances validated in spatial CV) and vertical extrapolation (outside the training elevation range). Exports go to ArcGIS Pro formats.

**Interpolation vs generalisation.** External boreholes in the *interpolation* group have neighbours in the development set within 100 m. Those in the *generalisation* group belong to isolated sites at least 500 m from any other borehole. The two groups are reported separately.

In [ ]:
import os, sys, json
from pathlib import Path

# Run mode: "full" for the experiment, "smoke" for the reduced software validation run.
# os.environ["SIEMREAP_MODE"] = "smoke"
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "config" / "project.toml").exists())
sys.path.insert(0, str(ROOT / "src"))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display, Image, Markdown

from siemreap import config
cfg = config.load_config(ROOT)
MODE = config.run_mode()
print(config.smoke_banner())

import joblib, hashlib
from siemreap import nested_cv as NCV, metrics as M, plots, grid3d, geo
AGG = config.stage_dir(cfg, "05_final", "aggregate")
RES = config.stage_dir(cfg, "05_final", "restricted")
mt = pd.read_csv(config.processed_dir(cfg) / "cells_all.csv")
external = pd.read_csv(config.processed_dir(cfg) / "external_holdout.csv").set_index("BH_Code")
selected = json.loads((config.stage_dir(cfg, "04_results", "aggregate") / "selected_model.json").read_text())
print(json.dumps(selected, indent=1))

## 1. Final model

In [ ]:
sel = selected["selected"]
# A documented override is possible here (e.g. after reviewing notebook 04), but it must be reported:
OVERRIDE = None        # e.g. {"scenario": "A", "model": "GWRF", "block_m": 100, "reason": "..."}
if OVERRIDE:
    sel = OVERRIDE
final = NCV.fit_final(mt, cfg, sel["scenario"], sel["model"], int(sel["block_m"]), MODE)
print({k: final[k] for k in ("scenario", "model_name", "block_m", "params", "threshold", "cv_macro_f1",
                             "n_train_boreholes", "n_train_cells")})

## 2. Freeze

In [ ]:
model_path = RES / f"final_model_{sel['scenario']}_{sel['model']}_{sel['block_m']}m.joblib"
joblib.dump(final["model"], model_path, compress=3)
model_sha = config.sha256_file(model_path)
frozen = {"scenario": sel["scenario"], "model": sel["model"], "block_m": int(sel["block_m"]), "params": final["params"],
          "threshold": final["threshold"], "cv_macro_f1_all_development": final["cv_macro_f1"],
          "n_train_boreholes": final["n_train_boreholes"], "n_train_cells": final["n_train_cells"],
          "top_elevation_range_m": final["top_elevation_range"], "model_file": model_path.name, "model_sha256": model_sha,
          "external_lock_sha256": json.loads((ROOT / "config" / "external_holdout.lock.json").read_text())["sha256"],
          "override": OVERRIDE, "mode": MODE, "environment": config.environment_record()}
(AGG / "final_model.json").write_text(json.dumps(frozen, indent=1, default=str))
final["trials"].to_csv(AGG / "final_model_optuna_trials.csv", index=False)
print("frozen:", model_path.name, model_sha[:16])

## 3. External evaluation (once, after freezing)

In [ ]:
ALLOW_REEVALUATION = False      # set True only with a documented reason; report it in the thesis
record_path = AGG / "external_evaluation.json"
if MODE == "smoke":
    # never read the external labels in a software test: use a stand-in of development boreholes
    rng = np.random.default_rng(0)
    dev_codes = mt[mt.in_A].BH_Code.drop_duplicates().sort_values().values
    stand_in = rng.choice(dev_codes, 8, replace=False)
    eval_cells = mt[mt.BH_Code.isin(stand_in)].assign(group="smoke_stand_in")
    label = "SMOKE STAND-IN (development boreholes; not an external evaluation)"
else:
    if record_path.exists():
        prev = json.loads(record_path.read_text())
        if prev["model_sha256"] != model_sha and not ALLOW_REEVALUATION:
            raise RuntimeError("The external set was already evaluated with a different frozen model. "
                               "Re-evaluation changes the meaning of the external test; set ALLOW_REEVALUATION only "
                               "with a documented reason.")
    eval_cells = mt[mt.set == "external"].assign(group=lambda d: d.BH_Code.map(external.group))
    label = "EXTERNAL HOLDOUT"
X = eval_cells[["UTM_Easting_m", "UTM_Northing_m", "Top_Elevation_m"]].values
p = final["model"].predict_proba(X)[:, 1]
ev = eval_cells.assign(p_clay=p, y_pred=(p >= final["threshold"]).astype(int), threshold=final["threshold"])
ev["dist_nearest_train_m"] = geo.nearest_distance(ev[["UTM_Easting_m", "UTM_Northing_m"]].values, final["train_bh_xy"])
ev["outer_fold"] = 0
rows = [{"group": "all", **M.metric_row(ev.y, ev.p_clay, ev.y_pred)}]
for g, d in ev.groupby("group"):
    rows.append({"group": g, **M.metric_row(d.y, d.p_clay, d.y_pred)})
ext = pd.DataFrame(rows)
print(label); display(ext.round(3))
ev.to_csv(RES / "external_predictions.csv", index=False)
(record_path).write_text(json.dumps({"label": label, "model_sha256": model_sha, "metrics": ext.to_dict("records"),
                                     "mode": MODE}, indent=1, default=str))
for k in ("scenario", "block_m", "purge_m", "model"):
    ev[k] = {"scenario": sel["scenario"], "block_m": sel["block_m"], "purge_m": 0.0, "model": sel["model"]}[k]
display(M.borehole_table(ev)[["BH_Code", "n", "clay_share", "accuracy", "macro_f1", "dist_nearest_train_m"]].round(3))
fig = plots.profiles(ev.assign(outer_fold=0), ev.BH_Code.drop_duplicates(), title=f"{label}: observed vs predicted")
fig.savefig(RES / "fig_external_profiles.png", bbox_inches="tight")

## 4. 3D prediction for ArcGIS Pro

**Horizontal flag.** A cell is flagged as horizontal extrapolation when its column is farther from the nearest training borehole than the 95th percentile of the test-to-nearest-training distances actually evaluated in the outer CV of the primary block size. Beyond that distance the CV says nothing about accuracy. The prediction-distance analysis showed that a large share of Downtown lies beyond it.

If the GP is the final model, predicting the full 25 m grid takes about 30–60 min; the tree models take minutes.

In [ ]:
preds = pd.read_csv(config.stage_dir(cfg, "03_nested_cv", "restricted") / "outer_predictions.csv")
pp = preds[(preds.block_m == sel["block_m"]) & (preds.purge_m == 0) & (preds.scenario == sel["scenario"])]
validated_m = float(pp.drop_duplicates(["BH_Code", "model"]).dist_nearest_train_m.quantile(0.95))
print(f"validated horizontal distance (P95 of outer-CV test-to-train distances): {validated_m:.0f} m")

gcfg = {**cfg, "grid3d": {**cfg["grid3d"], "spacing_m": cfg["smoke"]["grid_spacing_m"] if MODE == "smoke" else cfg["grid3d"]["spacing_m"]}}
cols = grid3d.columns(gcfg)
grid = grid3d.cells(cols, gcfg)
grid = grid3d.predict(final["model"], final["threshold"], grid, cfg["grid3d"]["chunk_rows"])
grid = grid3d.add_flags(grid, cols, final["train_bh_xy"], validated_m, final["top_elevation_range"])
print(f"{len(cols):,} columns x {grid.layer.nunique()} layers = {len(grid):,} cells")
share = {"horizontal_extrapolation": float(grid.extrap_horizontal.mean()), "vertical_extrapolation": float(grid.extrap_vertical.mean()),
         "predicted_clay_share": float(grid.clay.mean())}
print(json.dumps(share, indent=1))

In [ ]:
csv_path = RES / "siemreap_3d_points.csv"
nc_path = RES / "siemreap_3d_voxels.nc"
grid3d.to_points_csv(grid, csv_path)
vox = grid3d.to_voxel_netcdf(grid, cols, gcfg, nc_path)
summary3d = {"columns": int(len(cols)), "cells": int(len(grid)), "grid_spacing_m": gcfg["grid3d"]["spacing_m"],
             "validated_distance_m": validated_m, **share, "voxel": vox, "model_sha256": model_sha, "mode": MODE,
             "by_depth_band": grid.assign(band=pd.cut(grid.depth_top_m, cfg["depth"]["bands_m"], right=False).astype(str))
                                 .groupby("band")[["clay", "entropy", "extrap_horizontal"]].mean().round(3).to_dict("index")}
(AGG / "prediction_3d_summary.json").write_text(json.dumps(summary3d, indent=1, default=str))
print(json.dumps({k: v for k, v in summary3d.items() if k != "by_depth_band"}, indent=1, default=str))

In [ ]:
# Quick look: predicted class at three depths (restricted figure: Downtown extent)
fig, axes = plt.subplots(1, 3, figsize=(15, 4.2), dpi=150, sharey=True)
for ax, d in zip(axes, [1.0, 7.0, 13.0]):
    g = grid[np.isclose(grid.depth_top_m, d)]
    sc_ = ax.scatter(g.UTM_Easting_m, g.UTM_Northing_m, c=g.p_clay, cmap="RdBu", vmin=0, vmax=1, s=1 if MODE == "full" else 6, marker="s")
    h = g[g.extrap_horizontal == 1]
    ax.scatter(h.UTM_Easting_m, h.UTM_Northing_m, s=0.2, color="k", alpha=0.15)
    ax.set_title(f"P(Clay), depth {d:g}-{d + 0.5:g} m (dots: beyond validated distance)", fontsize=9, loc="left")
    ax.set_aspect("equal"); ax.ticklabel_format(useOffset=False, style="plain")
fig.colorbar(sc_, ax=axes, shrink=0.8, label="P(Clay): blue = Clay, red = Sand")
fig.savefig(RES / "fig_3d_depth_slices.png", bbox_inches="tight")

### Loading the outputs in ArcGIS Pro

* **3D points:** run *XY Table To Point* on `siemreap_3d_points.csv` with X Field = `X`, Y Field = `Y`, Z Field = `Z` and coordinate system WGS 1984 UTM Zone 48N (EPSG:32648). Symbolise by `p_clay` or `clay`, and filter on `extrap_horizontal` and `extrap_vertical`.
* **Voxel layer:** in a local scene, use *Add Data → Multidimensional → Add Voxel Layer* with `siemreap_3d_voxels.nc`. Variables are `p_clay`, `entropy`, `clay`, `extrap_horizontal` and `extrap_vertical`. Elevations are relative to the DEM's (unspecified) vertical datum.
* Both files are in `outputs/05_final/restricted/`, which is git-ignored.